# Chapter 8: Capstone: A Local Document Assistant

*Small Language Models in Practice — Haji Gul*

> Combining everything — a quantized model, RAG over your documents, a tool-using
agent, and a served endpoint — into one local assistant that answers questions
about your files and can do arithmetic on the way. This is the whole book in one
program.

---

*Lecture notes mirroring the book. Run the setup cell, then work top-to-bottom. Swap model ids freely.*

## Setup
Uncomment what this chapter needs.

In [ ]:
# %pip install -q transformers datasets accelerate torch
# Chapter-specific installs appear in shell cells below.

## What we are building

A single service that:

[leftmargin=1.4em]
 - loads a **quantized** model (Chapter 6) so it fits modest hardware;
 - retrieves context from your documents via **RAG** (Chapter 4);
 - can call a **tool** when the question needs computation (Chapter 5);
 - is exposed over **HTTP** (Chapter 7).

We assemble it from the pieces you already wrote, so each block should look
familiar.

## Part 1: the quantized model

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline,
)

quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map="auto")
gen = pipeline("text-generation", model=model, tokenizer=tok)

## Part 2: the retrieval index

In [ ]:
import lancedb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def build_index(texts):
    vecs = embedder.encode(texts, normalize_embeddings=True)
    db = lancedb.connect("./capstone_db")
    rows = [{"text": t, "vector": v.tolist()} for t, v in zip(texts, vecs)]
    return db.create_table("kb", data=rows, mode="overwrite")

# Replace with the chunks of your own documents.
table = build_index([
    "Quantization to 4-bit lets a 7B model run in about 3.5 GB.",
    "LoRA trains small adapters, updating under 1% of parameters.",
    "RAG injects retrieved document chunks into the prompt at query time.",
])

def retrieve(question, k=3):
    qv = embedder.encode(question, normalize_embeddings=True)
    return [h["text"] for h in table.search(qv.tolist()).limit(k).to_list()]

## Part 3: the tool

In [ ]:
import json

def calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    return str(eval(expression)) if set(expression) <= allowed else "invalid"

TOOLS = {"calculator": calculator}

## Part 4: the orchestrator

This is the brain: retrieve context, let the model optionally call the tool, then
produce a grounded final answer.

In [ ]:
SYSTEM = """You answer questions about the user's documents.
Use the provided context. If a calculation is needed, reply with ONLY:
{"tool": "calculator", "args": {"expression": "..."}}
Otherwise reply with a normal answer."""

def assistant(question):
    context = "\n".join(retrieve(question))
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",
         "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    for _ in range(3):
        reply = gen(messages, max_new_tokens=160,
                    do_sample=False)[0]["generated_text"][-1]["content"].strip()
        try:
            call = json.loads(reply)
        except json.JSONDecodeError:
            return reply                      # final grounded answer
        result = TOOLS[call["tool"]](**call["args"])
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"Tool result: {result}"})
    return reply

In [ ]:
print(assistant("How much memory does a 4-bit 7B model need?"))
print(assistant("If LoRA updates 0.8% of 1.5 billion params, how many is that? "
                "Use the calculator."))

## Part 5: serve it

Wrap the `assistant` function in the FastAPI pattern from Chapter 7 and
you have a deployable local document assistant.

In [ ]:
# file: app.py
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Local Document Assistant")

class Ask(BaseModel):
    question: str

@app.post("/ask")
def ask(a: Ask):
    return {"answer": assistant(a.question)}

In [ ]:
%%bash
uvicorn app:app --port 8000
curl -X POST localhost:8000/ask -H "Content-Type: application/json" \
  -d '{"question": "What does RAG do?"}'

> **You built the whole stack.** Quantized model + retrieval + tools + serving — the four pillars of practical
SLM work, in one file. Every production system is an elaboration of these same
pieces: better chunking, more tools, sturdier serving, evaluation, and
monitoring.

## Where to go next

[leftmargin=1.4em]
 - **Evaluation.** Build a small test set and score answers, so changes
 are measured, not guessed.
 - **Better retrieval.** Add re-ranking and metadata filters.
 - **Fine-tune the format.** Use Chapter 3 to lock the assistant's tone
 and structure.
 - **Scale serving.** Move from FastAPI to vLLM when traffic grows.

**Final exercise.** Replace the toy knowledge base with a folder of your own
documents, add one more tool relevant to your domain, and deploy the assistant
locally. You now have a private, controllable AI application that runs entirely
on your own hardware — the goal we set in the preface.